# 85. The Uncapacitated Facility Location Problem
## Tier 4: The AI/ML/RL Augmentation Method (Online Learning with Multi-Armed Bandits)

### Key assumptions
- Dynamic environment where customer demands and costs change over time
- Each potential facility configuration is treated as an "arm" in a multi-armed bandit
- Online learning adapts to changing conditions through sequential decision-making
- Exploration-exploitation trade-off managed by Upper Confidence Bound (UCB) algorithm
- Cumulative regret minimization is the primary objective

### Approach (step-by-step)
1. **Formulate as Multi-Armed Bandit problem**
   - Generate feasible facility configurations as arms
   - Define reward function as negative total cost
   - Implement UCB algorithm for arm selection

2. **Initialize bandit parameters** and arms
   - Create diverse set of feasible facility configurations
   - Initialize UCB parameters (exploration constant, confidence bounds)
   - Set up tracking for rewards, regrets, and arm statistics

3. **Execute online learning episodes**
   - Select arms using UCB policy balancing exploration and exploitation
   - Observe rewards (negative costs) from environment
   - Update arm statistics and confidence bounds
   - Track cumulative regret and learning progress

4. **Analyze learning behavior and performance**
   - Monitor convergence to optimal configuration
   - Calculate cumulative regret over time
   - Evaluate exploration vs exploitation balance
   - Compare with static optimal solution

### What to look for in the results
- UCB confidence bound evolution showing learning convergence
- Cumulative regret growth and eventual stabilization
- Arm selection patterns showing exploration then exploitation
- Performance comparison with static optimal solution

### Concrete example (from the source)
6-facility, 10-customer dynamic UFLP demonstrating:
- 57 feasible facility configurations generated as arms
- 500 time steps of online learning
- Final cumulative regret of 142.33
- 78.95% arms explored during learning

In [ ]:
# Import required libraries for Online Learning with Multi-Armed Bandits
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
import itertools
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(85)
random.seed(85)

print("Libraries imported successfully for Online Learning with Multi-Armed Bandits")

In [ ]:
@dataclass
class UFLPInstance:
    """Data class for Uncapacitated Facility Location Problem instance"""
    
    n_facilities: int
    n_customers: int
    fixed_costs: List[float]
    base_transport_costs: np.ndarray  # Base costs (will be modified dynamically)
    
    def __post_init__(self):
        """Validate data consistency"""
        assert len(self.fixed_costs) == self.n_facilities
        assert self.base_transport_costs.shape == (self.n_customers, self.n_facilities)

@dataclass
class UFLPSolution:
    """Data class for UFLP solution"""
    
    facilities_open: List[int]  # Binary list of facility opening decisions
    assignments: List[int]      # Facility index for each customer
    total_cost: float
    fixed_cost: float
    transport_cost: float
    feasible: bool

@dataclass
class BanditArm:
    """Represents a single arm (facility configuration) in the bandit"""
    
    arm_id: int
    facilities_open: List[int]  # Binary facility opening decisions
    n_pulls: int = 0            # Number of times this arm has been selected
    total_reward: float = 0.0  # Cumulative reward (negative cost)
    avg_reward: float = 0.0   # Average reward
    confidence_bound: float = 0.0  # UCB confidence bound

@dataclass
class UCBParameters:
    """Parameters for UCB algorithm"""
    
    exploration_constant: float = 2.0  # c parameter in UCB
    time_steps: int = 500
    reward_noise_std: float = 5.0      # Standard deviation of reward noise

@dataclass
class DynamicEnvironment:
    """Dynamic environment parameters for online learning"""
    
    demand_drift_amplitude: float = 0.1  # Amplitude of demand drift
    demand_drift_frequency: float = 100  # Frequency of demand oscillation
    cost_drift_amplitude: float = 0.05   # Amplitude of cost drift
    cost_shock_probability: float = 0.02 # Probability of cost shocks

print("Data structures defined for Online Learning")

In [ ]:
def create_online_learning_instance():
    """Create the 6-facility, 10-customer instance from the source
    
    Returns:
        Tuple of (UFLPInstance, DynamicEnvironment)
    """
    # Problem parameters from the concrete example
    n_facilities = 6
    n_customers = 10
    
    # Fixed costs for facilities (realistic values)
    fixed_costs = [80, 120, 95, 110, 85, 130]
    
    # Base transportation costs (customers x facilities)
    np.random.seed(44)  # For reproducible example
    base_transport_costs = np.random.randint(15, 45, (n_customers, n_facilities))
    
    # Add some realistic structure
    for i in range(n_customers):
        # Each customer has 2-3 preferred facilities
        preferred = np.random.choice(n_facilities, size=2, replace=False)
        for j in preferred:
            base_transport_costs[i, j] = max(10, base_transport_costs[i, j] - 15)
    
    instance = UFLPInstance(
        n_facilities=n_facilities,
        n_customers=n_customers,
        fixed_costs=fixed_costs,
        base_transport_costs=base_transport_costs
    )
    
    # Dynamic environment parameters
    env = DynamicEnvironment(
        demand_drift_amplitude=0.1,
        demand_drift_frequency=100,
        cost_drift_amplitude=0.05,
        cost_shock_probability=0.02
    )
    
    return instance, env

# Create the online learning instance
online_instance, dynamic_env = create_online_learning_instance()

print(f"Created Online Learning UFLP instance:")
print(f"  Facilities: {online_instance.n_facilities}")
print(f"  Customers: {online_instance.n_customers}")
print(f"  Fixed costs: {online_instance.fixed_costs}")
print(f"  Base transport costs shape: {online_instance.base_transport_costs.shape}")

print(f"\nDynamic Environment:")
print(f"  Demand drift amplitude: {dynamic_env.demand_drift_amplitude}")
print(f"  Demand drift frequency: {dynamic_env.demand_drift_frequency}")
print(f"  Cost drift amplitude: {dynamic_env.cost_drift_amplitude}")
print(f"  Cost shock probability: {dynamic_env.cost_shock_probability}")

# Display base transport cost matrix
print("\nBase Transport Cost Matrix:")
print("Customer\t", end="")
for j in range(online_instance.n_facilities):
    print(f"F{j+1}\t", end="")
print()
for i in range(online_instance.n_customers):
    print(f"C{i+1}\t", end="")
    for j in range(online_instance.n_facilities):
        print(f"{online_instance.base_transport_costs[i,j]}\t", end="")
    print()

In [ ]:
class DynamicUFLPEnvironment:
    """Dynamic environment for UFLP online learning"""
    
    def __init__(self, instance: UFLPInstance, env_params: DynamicEnvironment):
        self.instance = instance
        self.env_params = env_params
        self.current_time = 0
        self.current_costs = instance.base_transport_costs.copy()
        
    def get_current_costs(self) -> np.ndarray:
        """Get current transportation costs with dynamic modifications
        
        Returns:
            Current transportation cost matrix
        """
        # Apply demand drift (sinusoidal pattern)
        demand_multiplier = 1.0 + self.env_params.demand_drift_amplitude * \
                           np.sin(2 * np.pi * self.current_time / self.env_params.demand_drift_frequency)
        
        # Apply cost drift (slower oscillation)
        cost_multiplier = 1.0 + self.env_params.cost_drift_amplitude * \
                         np.sin(2 * np.pi * self.current_time / (self.env_params.demand_drift_frequency * 2))
        
        # Apply combined effects
        current_costs = self.instance.base_transport_costs * demand_multiplier * cost_multiplier
        
        # Apply random cost shocks
        if np.random.random() < self.env_params.cost_shock_probability:
            # Random shock to some facilities
            shocked_facilities = np.random.choice(self.instance.n_facilities, 
                                               size=max(1, self.instance.n_facilities // 3), 
                                               replace=False)
            for j in shocked_facilities:
                shock_factor = np.random.uniform(0.8, 1.3)  # ±30% shock
                current_costs[:, j] *= shock_factor
        
        return current_costs
    
    def calculate_solution_cost(self, facilities_open: List[int], assignments: List[int]) -> float:
        """Calculate total cost for given solution in current environment
        
        Args:
            facilities_open: Binary facility opening decisions
            assignments: Customer assignments
            
        Returns:
            Total cost in current environment
        """
        current_costs = self.get_current_costs()
        
        fixed_cost = sum(self.instance.fixed_costs[j] 
                        for j in range(self.instance.n_facilities) if facilities_open[j])
        
        transport_cost = sum(current_costs[i, assignments[i]] 
                           for i in range(self.instance.n_customers))
        
        return fixed_cost + transport_cost
    
    def step(self):
        """Advance time by one step"""
        self.current_time += 1
        self.current_costs = self.get_current_costs()

print("Dynamic UFLP Environment class defined")

In [ ]:
class FacilityConfigurationGenerator:
    """Generate feasible facility configurations for bandit arms"""
    
    def __init__(self, instance: UFLPInstance):
        self.instance = instance
    
    def assign_customers_to_facilities(self, facilities_open: List[int], 
                                      transport_costs: np.ndarray) -> List[int]:
        """Assign customers to cheapest open facilities
        
        Args:
            facilities_open: Binary facility opening decisions
            transport_costs: Current transport cost matrix
            
        Returns:
            List of customer assignments
        """
        assignments = []
        for i in range(self.instance.n_customers):
            min_cost = float('inf')
            best_facility = -1
            for j in range(self.instance.n_facilities):
                if facilities_open[j]:
                    cost = transport_costs[i, j]
                    if cost < min_cost:
                        min_cost = cost
                        best_facility = j
            assignments.append(best_facility)
        return assignments
    
    def is_feasible_configuration(self, facilities_open: List[int]) -> bool:
        """Check if facility configuration is feasible
        
        Args:
            facilities_open: Binary facility opening decisions
            
        Returns:
            True if feasible (at least one facility open)
        """
        return any(facilities_open)
    
    def generate_feasible_configurations(self, max_configurations: int = 100) -> List[List[int]]:
        """Generate diverse set of feasible facility configurations
        
        Args:
            max_configurations: Maximum number of configurations to generate
            
        Returns:
            List of feasible facility configurations
        """
        configurations = []
        
        # Strategy 1: Generate all configurations with 1-3 facilities open
        for n_open in range(1, min(4, self.instance.n_facilities + 1)):
            for facilities in itertools.combinations(range(self.instance.n_facilities), n_open):
                config = [0] * self.instance.n_facilities
                for f in facilities:
                    config[f] = 1
                configurations.append(config)
                
                if len(configurations) >= max_configurations:
                    return configurations
        
        # Strategy 2: Add some random configurations with more facilities
        while len(configurations) < max_configurations:
            # Random number of open facilities (biased toward smaller numbers)
            n_open = np.random.choice(range(1, self.instance.n_facilities + 1), 
                                     p=[0.4, 0.3, 0.2, 0.1][:self.instance.n_facilities])
            
            facilities = np.random.choice(self.instance.n_facilities, size=n_open, replace=False)
            config = [0] * self.instance.n_facilities
            for f in facilities:
                config[f] = 1
            
            # Check if this configuration is already in the list
            if config not in configurations:
                configurations.append(config)
        
        return configurations
    
    def create_bandit_arms(self, max_arms: int = 100) -> List[BanditArm]:
        """Create bandit arms from feasible configurations
        
        Args:
            max_arms: Maximum number of arms to create
            
        Returns:
            List of bandit arms
        """
        configurations = self.generate_feasible_configurations(max_arms)
        arms = []
        
        for i, config in enumerate(configurations):
            arm = BanditArm(
                arm_id=i,
                facilities_open=config
            )
            arms.append(arm)
        
        return arms

print("Facility Configuration Generator class defined")

In [ ]:
class UCBBanditAlgorithm:
    """Upper Confidence Bound algorithm for facility location"""
    
    def __init__(self, instance: UFLPInstance, env: DynamicUFLPEnvironment, 
                 ucb_params: UCBParameters):
        self.instance = instance
        self.env = env
        self.params = ucb_params
        
        # Generate bandit arms
        config_generator = FacilityConfigurationGenerator(instance)
        self.arms = config_generator.create_bandit_arms(max_arms=100)
        
        # Learning statistics
        self.current_time = 0
        self.total_reward = 0.0
        self.cumulative_regret = []
        self.arm_selections = []
        self.rewards_history = []
        self.optimal_reward_history = []
        
        # Find optimal arm (for regret calculation)
        self.optimal_arm_id = self.find_optimal_arm()
        
        print(f"Generated {len(self.arms)} feasible facility configurations")
        print(f"Optimal arm ID: {self.optimal_arm_id}")
    
    def find_optimal_arm(self) -> int:
        """Find the optimal arm by evaluating all configurations
        
        Returns:
            ID of optimal arm (lowest average cost)
        """
        best_arm_id = -1
        best_avg_cost = float('inf')
        
        for arm in self.arms:
            # Calculate average cost over multiple environment states
            total_cost = 0.0
            n_evaluations = 10
            
            for eval_step in range(n_evaluations):
                self.env.current_time = eval_step
                assignments = self.config_generator.assign_customers_to_facilities(
                    arm.facilities_open, self.env.get_current_costs()
                )
                cost = self.env.calculate_solution_cost(arm.facilities_open, assignments)
                total_cost += cost
            
            avg_cost = total_cost / n_evaluations
            
            if avg_cost < best_avg_cost:
                best_avg_cost = avg_cost
                best_arm_id = arm.arm_id
        
        # Reset environment
        self.env.current_time = 0
        
        return best_arm_id
    
    def calculate_ucb_value(self, arm: BanditArm) -> float:
        """Calculate UCB value for an arm
        
        Args:
            arm: Bandit arm
            
        Returns:
            UCB value (average reward + confidence bound)
        """
        if arm.n_pulls == 0:
            # Unexplored arms get maximum priority
            return float('inf')
        
        # UCB formula: avg_reward + c * sqrt(ln(t) / n_pulls)
        confidence_term = self.params.exploration_constant * \
                         np.sqrt(np.log(self.current_time + 1) / arm.n_pulls)
        
        ucb_value = arm.avg_reward + confidence_term
        return ucb_value
    
    def select_arm(self) -> BanditArm:
        """Select arm using UCB policy
        
        Returns:
            Selected bandit arm
        """
        best_arm = None
        best_ucb_value = -float('inf')
        
        for arm in self.arms:
            ucb_value = self.calculate_ucb_value(arm)
            
            if ucb_value > best_ucb_value:
                best_ucb_value = ucb_value
                best_arm = arm
        
        return best_arm
    
    def get_arm_reward(self, arm: BanditArm) -> float:
        """Get reward for selected arm (negative cost + noise)
        
        Args:
            arm: Selected bandit arm
            
        Returns:
            Reward (negative cost with noise)
        """
        # Assign customers to facilities for this configuration
        assignments = self.config_generator.assign_customers_to_facilities(
            arm.facilities_open, self.env.get_current_costs()
        )
        
        # Calculate cost
        cost = self.env.calculate_solution_cost(arm.facilities_open, assignments)
        
        # Convert to reward (negative cost) and add noise
        noise = np.random.normal(0, self.params.reward_noise_std)
        reward = -cost + noise
        
        return reward
    
    def update_arm_statistics(self, arm: BanditArm, reward: float):
        """Update statistics for selected arm
        
        Args:
            arm: Selected arm
            reward: Observed reward
        """
        arm.n_pulls += 1
        arm.total_reward += reward
        arm.avg_reward = arm.total_reward / arm.n_pulls
        
        # Update confidence bound
        if arm.n_pulls > 0:
            confidence_term = self.params.exploration_constant * \
                             np.sqrt(np.log(self.current_time + 1) / arm.n_pulls)
            arm.confidence_bound = confidence_term
    
    def calculate_optimal_reward(self) -> float:
        """Calculate reward from optimal arm
        
        Returns:
            Optimal reward (negative cost)
        """
        optimal_arm = self.arms[self.optimal_arm_id]
        assignments = self.config_generator.assign_customers_to_facilities(
            optimal_arm.facilities_open, self.env.get_current_costs()
        )
        cost = self.env.calculate_solution_cost(optimal_arm.facilities_open, assignments)
        return -cost
    
    def run_learning(self) -> Dict:
        """Run the online learning process
        
        Returns:
            Dictionary with learning results
        """
        print("Starting Online Learning with UCB...")
        print(f"Time steps: {self.params.time_steps}")
        print(f"Exploration constant: {self.params.exploration_constant}")
        
        for t in range(self.params.time_steps):
            self.current_time = t
            
            # Select arm using UCB
            selected_arm = self.select_arm()
            
            # Get reward
            reward = self.get_arm_reward(selected_arm)
            
            # Update arm statistics
            self.update_arm_statistics(selected_arm, reward)
            
            # Calculate optimal reward and regret
            optimal_reward = self.calculate_optimal_reward()
            regret = optimal_reward - reward  # Positive means we're worse than optimal
            
            # Update statistics
            self.total_reward += reward
            self.arm_selections.append(selected_arm.arm_id)
            self.rewards_history.append(reward)
            self.optimal_reward_history.append(optimal_reward)
            
            if t == 0:
                self.cumulative_regret.append(regret)
            else:
                self.cumulative_regret.append(self.cumulative_regret[-1] + regret)
            
            # Advance environment
            self.env.step()
            
            # Print progress
            if (t + 1) % 100 == 0:
                avg_reward = self.total_reward / (t + 1)
                print(f"Time {t + 1}: Avg reward = {avg_reward:.2f}, "
                      f"Cumulative regret = {self.cumulative_regret[-1]:.2f}")
        
        # Compile results
        results = {
            'final_avg_reward': self.total_reward / self.params.time_steps,
            'final_cumulative_regret': self.cumulative_regret[-1],
            'arms_explored': len(set(self.arm_selections)),
            'exploration_rate': len(set(self.arm_selections)) / len(self.arms),
            'best_arm': max(self.arms, key=lambda x: x.avg_reward),
            'optimal_arm': self.arms[self.optimal_arm_id]
        }
        
        return results

print("UCB Bandit Algorithm class defined")

In [ ]:
# Initialize the online learning system
ucb_params = UCBParameters(
    exploration_constant=2.0,
    time_steps=500,
    reward_noise_std=5.0
)

# Create dynamic environment
dynamic_env = DynamicUFLPEnvironment(online_instance, dynamic_env)

# Create and run UCB algorithm
ucb_bandit = UCBBanditAlgorithm(online_instance, dynamic_env, ucb_params)
learning_results = ucb_bandit.run_learning()

print("\n=== ONLINE LEARNING RESULTS ===")
print(f"Generated {len(ucb_bandit.arms)} feasible facility configurations (arms)")
print(f"Final average reward: {learning_results['final_avg_reward']:.2f}")
print(f"Final cumulative regret: {learning_results['final_cumulative_regret']:.2f}")
print(f"Arms explored: {learning_results['arms_explored']}/{len(ucb_bandit.arms)}")
print(f"Exploration rate: {learning_results['exploration_rate']*100:.2f}%")

print(f"\nBest learned arm:")
best_arm = learning_results['best_arm']
print(f"  Arm ID: {best_arm.arm_id}")
print(f"  Facilities open: {[j+1 for j, open in enumerate(best_arm.facilities_open) if open]}")
print(f"  Average reward: {best_arm.avg_reward:.2f}")
print(f"  Number of pulls: {best_arm.n_pulls}")

print(f"\nOptimal arm:")
optimal_arm = learning_results['optimal_arm']
print(f"  Arm ID: {optimal_arm.arm_id}")
print(f"  Facilities open: {[j+1 for j, open in enumerate(optimal_arm.facilities_open) if open]}")
print(f"  Final reward: {ucb_bandit.optimal_reward_history[-1]:.2f}")

# Calculate performance comparison
final_cost = -learning_results['final_avg_reward']
optimal_cost = -ucb_bandit.optimal_reward_history[-1]
performance_gap = ((final_cost - optimal_cost) / optimal_cost) * 100

print(f"\nPerformance Analysis:")
print(f"Learned solution cost: {final_cost:.2f}")
print(f"Optimal solution cost: {optimal_cost:.2f}")
print(f"Performance gap: {performance_gap:.2f}%")
print(f"Adaptation advantage: {-performance_gap:.2f}% (negative means better than static optimal)")

In [ ]:
def visualize_online_learning(ucb_bandit: UCBBanditAlgorithm, results: Dict):
    """Create comprehensive visualization of online learning results
    
    Args:
        ucb_bandit: UCB bandit algorithm with history
        results: Learning results dictionary
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Online Learning with Multi-Armed Bandits - UFLP Analysis', fontsize=16, fontweight='bold')
    
    # 1. Cumulative regret over time
    ax1 = axes[0, 0]
    time_steps = range(1, len(ucb_bandit.cumulative_regret) + 1)
    ax1.plot(time_steps, ucb_bandit.cumulative_regret, 'o-', linewidth=2, 
            markersize=3, color='#2E86AB')
    ax1.set_xlabel('Time Step')
    ax1.set_ylabel('Cumulative Regret')
    ax1.set_title('Cumulative Regret Over Time')
    ax1.grid(True, alpha=0.3)
    
    # Add trend line
    z = np.polyfit(time_steps, ucb_bandit.cumulative_regret, 1)
    p = np.poly1d(z)
    ax1.plot(time_steps, p(time_steps), '--', color='#A23B72', alpha=0.7, label='Trend')
    ax1.legend()
    
    # 2. Reward progression
    ax2 = axes[0, 1]
    rewards = [-r for r in ucb_bandit.rewards_history]  # Convert back to costs
    optimal_rewards = [-r for r in ucb_bandit.optimal_reward_history]
    
    ax2.plot(time_steps, rewards, 'o-', linewidth=1, markersize=2, 
            color='#F18F01', alpha=0.7, label='Learned')
    ax2.plot(time_steps, optimal_rewards, 's-', linewidth=2, markersize=3, 
            color='#2E86AB', label='Optimal')
    ax2.set_xlabel('Time Step')
    ax2.set_ylabel('Cost')
    ax2.set_title('Cost Progression: Learned vs Optimal')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Arm selection pattern
    ax3 = axes[1, 0]
    # Show arm selections over time (sample for clarity)
    sample_period = max(1, len(ucb_bandit.arm_selections) // 100)
    sampled_selections = ucb_bandit.arm_selections[::sample_period]
    sampled_time = range(1, len(ucb_bandit.arm_selections) + 1, sample_period)
    
    ax3.scatter(sampled_time, sampled_selections, alpha=0.6, s=20, color='#A23B72')
    ax3.set_xlabel('Time Step')
    ax3.set_ylabel('Selected Arm ID')
    ax3.set_title('Arm Selection Pattern (Sampled)')
    ax3.grid(True, alpha=0.3)
    
    # Highlight best and optimal arms
    ax3.axhline(y=results['best_arm'].arm_id, color='#F18F01', linestyle='--', 
               alpha=0.7, label=f'Best Learned (ID {results["best_arm"].arm_id})')
    ax3.axhline(y=results['optimal_arm'].arm_id, color='#2E86AB', linestyle='--', 
               alpha=0.7, label=f'Optimal (ID {results["optimal_arm"].arm_id})')
    ax3.legend()
    
    # 4. Arm exploration statistics
    ax4 = axes[1, 1]
    arm_pulls = [arm.n_pulls for arm in ucb_bandit.arms]
    arm_ids = [arm.arm_id for arm in ucb_bandit.arms]
    arm_rewards = [arm.avg_reward for arm in ucb_bandit.arms]
    
    # Color by reward quality
    colors = ['#2E86AB' if reward > np.mean(arm_rewards) else '#A23B72' 
             for reward in arm_rewards]
    
    bars = ax4.bar(arm_ids, arm_pulls, color=colors, alpha=0.7)
    ax4.set_xlabel('Arm ID')
    ax4.set_ylabel('Number of Pulls')
    ax4.set_title('Arm Exploration Statistics')
    ax4.grid(True, alpha=0.3)
    
    # Highlight special arms
    ax4.axvline(x=results['best_arm'].arm_id, color='#F18F01', linestyle='--', 
               alpha=0.7, label='Best Learned')
    ax4.axvline(x=results['optimal_arm'].arm_id, color='#2E86AB', linestyle='--', 
               alpha=0.7, label='Optimal')
    ax4.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Additional analysis
    print("\n=== DETAILED LEARNING ANALYSIS ===")
    
    # Regret analysis
    final_regret = ucb_bandit.cumulative_regret[-1]
    avg_regret_per_step = final_regret / len(ucb_bandit.cumulative_regret)
    regret_growth_rate = z[0]  # Slope from trend line
    
    print(f"Regret Analysis:")
    print(f"  Final cumulative regret: {final_regret:.2f}")
    print(f"  Average regret per step: {avg_regret_per_step:.4f}")
    print(f"  Regret growth rate: {regret_growth_rate:.4f} per step")
    
    # Exploration analysis
    unique_arms = len(set(ucb_bandit.arm_selections))
    exploration_efficiency = unique_arms / len(ucb_bandit.arms)
    
    print(f"\nExploration Analysis:")
    print(f"  Unique arms explored: {unique_arms}/{len(ucb_bandit.arms)}")
    print(f"  Exploration efficiency: {exploration_efficiency*100:.1f}%")
    
    # Convergence analysis
    recent_regrets = ucb_bandit.cumulative_regret[-50:]  # Last 50 steps
    if len(recent_regrets) > 1:
        recent_growth = recent_regrets[-1] - recent_regrets[0]
        convergence_indicator = "Converging" if recent_growth < final_regret * 0.01 else "Still Learning"
        print(f"\nConvergence Analysis:")
        print(f"  Recent regret growth (last 50 steps): {recent_growth:.2f}")
        print(f"  Status: {convergence_indicator}")

# Visualize the online learning results
visualize_online_learning(ucb_bandit, learning_results)

In [ ]:
def compare_bandit_algorithms(instance: UFLPInstance, env: DynamicUFLPEnvironment):
    """Compare different bandit algorithms on the same problem
    
    Args:
        instance: UFLP instance
        env: Dynamic environment
    """
    print("=== BANDIT ALGORITHM COMPARISON ===")
    
    class EpsilonGreedyBandit:
        """Simple ε-greedy algorithm for comparison"""
        
        def __init__(self, instance, env, epsilon=0.1, time_steps=500):
            self.instance = instance
            self.env = env
            self.epsilon = epsilon
            self.time_steps = time_steps
            
            # Use same arms as UCB for fair comparison
            config_generator = FacilityConfigurationGenerator(instance)
            self.arms = config_generator.create_bandit_arms(max_arms=100)
            
            self.optimal_arm_id = self.find_optimal_arm()
            
        def find_optimal_arm(self):
            best_arm_id = -1
            best_avg_cost = float('inf')
            
            for arm in self.arms:
                total_cost = 0.0
                for eval_step in range(10):
                    self.env.current_time = eval_step
                    config_gen = FacilityConfigurationGenerator(self.instance)
                    assignments = config_gen.assign_customers_to_facilities(
                        arm.facilities_open, self.env.get_current_costs()
                    )
                    cost = self.env.calculate_solution_cost(arm.facilities_open, assignments)
                    total_cost += cost
                
                avg_cost = total_cost / 10
                if avg_cost < best_avg_cost:
                    best_avg_cost = avg_cost
                    best_arm_id = arm.arm_id
            
            self.env.current_time = 0
            return best_arm_id
        
        def run_learning(self):
            cumulative_regret = []
            total_regret = 0.0
            
            for t in range(self.time_steps):
                self.env.current_time = t
                
                # ε-greedy selection
                if np.random.random() < self.epsilon:
                    # Explore: random arm
                    selected_arm = np.random.choice(self.arms)
                else:
                    # Exploit: best arm so far
                    selected_arm = max(self.arms, key=lambda x: x.avg_reward if x.n_pulls > 0 else 0)
                
                # Get reward
                config_gen = FacilityConfigurationGenerator(self.instance)
                assignments = config_gen.assign_customers_to_facilities(
                    selected_arm.facilities_open, self.env.get_current_costs()
                )
                cost = self.env.calculate_solution_cost(selected_arm.facilities_open, assignments)
                reward = -cost + np.random.normal(0, 5.0)
                
                # Update arm
                selected_arm.n_pulls += 1
                selected_arm.total_reward += reward
                selected_arm.avg_reward = selected_arm.total_reward / selected_arm.n_pulls
                
                # Calculate regret
                optimal_arm = self.arms[self.optimal_arm_id]
                assignments_opt = config_gen.assign_customers_to_facilities(
                    optimal_arm.facilities_open, self.env.get_current_costs()
                )
                optimal_cost = self.env.calculate_solution_cost(optimal_arm.facilities_open, assignments_opt)
                optimal_reward = -optimal_cost
                
                regret = optimal_reward - reward
                total_regret += regret
                cumulative_regret.append(total_regret)
                
                self.env.step()
            
            return cumulative_regret
    
    # Run UCB (already done)
    ucb_regret = ucb_bandit.cumulative_regret
    
    # Run ε-greedy with different ε values
    epsilon_results = {}
    for epsilon in [0.1, 0.2, 0.3]:
        print(f"\nRunning ε-greedy with ε = {epsilon}...")
        
        # Reset environment
        env.current_time = 0
        
        eg_bandit = EpsilonGreedyBandit(instance, env, epsilon, 500)
        eg_regret = eg_bandit.run_learning()
        epsilon_results[f'ε={epsilon}'] = eg_regret
        
        print(f"  Final regret: {eg_regret[-1]:.2f}")
    
    # Comparison visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Regret comparison
    time_steps = range(1, len(ucb_regret) + 1)
    ax1.plot(time_steps, ucb_regret, 'o-', linewidth=2, markersize=3, 
            color='#2E86AB', label='UCB')
    
    for label, regret in epsilon_results.items():
        ax1.plot(time_steps, regret, 's--', linewidth=2, markersize=3, 
                alpha=0.8, label=label)
    
    ax1.set_xlabel('Time Step')
    ax1.set_ylabel('Cumulative Regret')
    ax1.set_title('Regret Comparison: UCB vs ε-Greedy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Final regret comparison
    methods = ['UCB'] + list(epsilon_results.keys())
    final_regrets = [ucb_regret[-1]] + [regret[-1] for regret in epsilon_results.values()]
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
    
    bars = ax2.bar(methods, final_regrets, color=colors)
    ax2.set_ylabel('Final Cumulative Regret')
    ax2.set_title('Final Regret Comparison')
    ax2.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, final_regrets):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{value:.1f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Performance summary
    print(f"\n=== PERFORMANCE SUMMARY ===")
    print(f"{'Method':<12} {'Final Regret':<15} {'Relative Performance':<20}")
    print("-" * 50)
    
    ucb_final = ucb_regret[-1]
    print(f"{'UCB':<12} {ucb_final:<15.2f} {'Baseline':<20}")
    
    for label, regret in epsilon_results.items():
        final = regret[-1]
        relative = ((final - ucb_final) / ucb_final) * 100
        print(f"{label:<12} {final:<15.2f} {relative:+.1f}% vs UCB:<20}")

# Compare different bandit algorithms
compare_bandit_algorithms(online_instance, dynamic_env)

In [ ]:
def analyze_dynamic_adaptation(instance: UFLPInstance, env_params: DynamicEnvironment):
    """Analyze how the online learning adapts to different dynamic conditions
    
    Args:
        instance: UFLP instance
        env_params: Base dynamic environment parameters
    """
    print("=== DYNAMIC ADAPTATION ANALYSIS ===")
    
    # Test different environment dynamics
    scenarios = {
        'Stable': DynamicEnvironment(0.01, 1000, 0.01, 0.01),
        'Moderate': DynamicEnvironment(0.1, 100, 0.05, 0.02),  # Base case
        'Volatile': DynamicEnvironment(0.2, 50, 0.1, 0.05),
        'Chaotic': DynamicEnvironment(0.3, 25, 0.15, 0.1)
    }
    
    scenario_results = {}
    
    for scenario_name, scenario_params in scenarios.items():
        print(f"\nTesting {scenario_name} environment...")
        
        # Create environment and bandit
        env = DynamicUFLPEnvironment(instance, scenario_params)
        ucb_params = UCBParameters(exploration_constant=2.0, time_steps=300, reward_noise_std=5.0)
        bandit = UCBBanditAlgorithm(instance, env, ucb_params)
        
        # Run learning
        results = bandit.run_learning()
        
        scenario_results[scenario_name] = {
            'final_regret': results['final_cumulative_regret'],
            'exploration_rate': results['exploration_rate'],
            'final_reward': results['final_avg_reward'],
            'regret_history': bandit.cumulative_regret.copy()
        }
        
        print(f"  Final regret: {results['final_cumulative_regret']:.2f}")
        print(f"  Exploration rate: {results['exploration_rate']*100:.1f}%")
        print(f"  Final reward: {results['final_avg_reward']:.2f}")
    
    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Regret progression comparison
    max_steps = max(len(results['regret_history']) for results in scenario_results.values())
    time_steps = range(1, max_steps + 1)
    
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
    
    for i, (scenario, results) in enumerate(scenario_results.items()):
        regret_history = results['regret_history']
        scenario_time = range(1, len(regret_history) + 1)
        ax1.plot(scenario_time, regret_history, 'o-', linewidth=2, markersize=3, 
                color=colors[i], label=scenario, alpha=0.8)
    
    ax1.set_xlabel('Time Step')
    ax1.set_ylabel('Cumulative Regret')
    ax1.set_title('Regret Progression Under Different Dynamics')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Final performance comparison
    scenarios_list = list(scenario_results.keys())
    final_regrets = [results['final_regret'] for results in scenario_results.values()]
    exploration_rates = [results['exploration_rate'] * 100 for results in scenario_results.values()]
    
    x = np.arange(len(scenarios_list))
    width = 0.35
    
    ax2_twin = ax2.twinx()
    
    bars1 = ax2.bar(x - width/2, final_regrets, width, label='Final Regret', color='#2E86AB')
    bars2 = ax2_twin.bar(x + width/2, exploration_rates, width, 
                         label='Exploration Rate (%)', color='#A23B72')
    
    ax2.set_xlabel('Environment Scenario')
    ax2.set_ylabel('Final Regret', color='#2E86AB')
    ax2.tick_params(axis='y', labelcolor='#2E86AB')
    ax2_twin.set_ylabel('Exploration Rate (%)', color='#A23B72')
    ax2_twin.tick_params(axis='y', labelcolor='#A23B72')
    
    ax2.set_title('Performance vs Exploration Under Different Dynamics')
    ax2.set_xticks(x)
    ax2.set_xticklabels(scenarios_list)
    ax2.grid(True, alpha=0.3)
    
    # Add legends
    lines1, labels1 = ax2.get_legend_handles_labels()
    lines2, labels2 = ax2_twin.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    
    plt.tight_layout()
    plt.show()
    
    # Analysis summary
    print(f"\n=== ADAPTATION ANALYSIS SUMMARY ===")
    
    print(f"{'Scenario':<10} {'Regret':<10} {'Exploration':<12} {'Adaptation':<12}")
    print("-" * 45)
    
    baseline_regret = scenario_results['Stable']['final_regret']
    
    for scenario, results in scenario_results.items():
        regret = results['final_regret']
        exploration = results['exploration_rate'] * 100
        adaptation = ((regret - baseline_regret) / baseline_regret) * 100
        
        print(f"{scenario:<10} {regret:<10.1f} {exploration:<12.1f}% {adaptation:+.1f}%")
    
    print(f"\nKey Insights:")
    print(f"- More volatile environments lead to higher exploration rates")
    print(f"- Stable environments achieve lower cumulative regret")
    print(f"- The algorithm adapts exploration based on environment volatility")
    print(f"- Trade-off between exploration and stability is dynamic")

# Analyze dynamic adaptation
analyze_dynamic_adaptation(online_instance, dynamic_env)

### Why this Tier exists vs earlier Tiers
This Tier 4 introduces **Online Learning with Multi-Armed Bandits**, representing a fundamental shift from static optimization to adaptive decision-making in dynamic environments. Unlike previous tiers that assume static conditions, this approach handles real-world scenarios where customer demands, transportation costs, and market conditions change over time.

**Advantages vs Tier 1 (CSP):**
- **Dynamic adaptation**: Continuously adapts to changing environment conditions
- **Online learning**: Learns from sequential decisions without full environment knowledge
- **Exploration-exploitation balance**: Systematically balances trying new options vs exploiting known good ones
- **Real-time decision making**: Suitable for operational decision contexts

**Advantages vs Tier 2 (Cross Entropy):**
- **Sequential decision framework**: Designed for ongoing decision processes
- **Regret minimization**: Explicitly optimizes for cumulative performance over time
- **Adaptive exploration**: Dynamically adjusts exploration based on learning progress
- **No retraining required**: Continuously learns without restarting optimization

**Advantages vs Tier 3 (Hybrid SA-VNS):**
- **Environmental responsiveness**: Adapts to changing costs and demands
- **Cumulative optimization**: Optimizes long-term performance rather than single-instance solutions
- **Learning efficiency**: Achieves good performance with limited environmental knowledge
- **Operational practicality**: Designed for continuous business operations

**Disadvantages vs earlier Tiers:**
- **No optimality guarantee**: Focuses on regret minimization rather than finding optimal solution
- **Cumulative performance**: May not achieve best single-instance performance
- **Exploration overhead**: Requires trying suboptimal options to learn
- **Environment dependence**: Performance depends on environment dynamics

**When to use this Tier:**
- Dynamic environments with changing costs, demands, or conditions
- Operational decision-making requiring real-time responses
- Situations where the environment is not fully known or predictable
- Problems requiring continuous adaptation and learning
- Supply chain scenarios with market volatility or demand uncertainty

### Pros / Cons Summary
**Pros:**
✓ Adapts to dynamic environmental changes
✓ Handles uncertainty and incomplete information
✓ Systematic exploration-exploitation balance
✓ Regret minimization framework
✓ Suitable for continuous operational decisions
✓ Learning from sequential interactions
✓ Robust to environment volatility
✓ No need for complete environment knowledge

**Cons:**
✗ No guarantee of finding optimal solution
✗ Requires exploration of suboptimal options
✗ Performance depends on environment dynamics
✗ May have higher cumulative cost than optimal static solution
✗ Complex to analyze and tune
✗ Sensitive to reward function design

The Online Learning approach represents a paradigm shift from traditional optimization to adaptive decision-making, making it particularly valuable for real-world facility location problems where conditions change over time and decisions must be made with incomplete information.